# Gradient Descent: How Models Actually Learn

## CE 315 - Junior Design

In our last class, we used `.fit()` and it magically found the best line. But **how** did it find that line?

Today we'll peek under the hood and see the algorithm that powers most of machine learning: **Gradient Descent**.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set random seed for reproducibility
np.random.seed(42)

## Part 1: The Problem - Finding the Best Line

Let's start with simple data: predicting house prices from square footage.

In [ ]:
# Simple dataset
X = np.array([1, 2, 3, 4, 5]).reshape(-1, 1)  # Square feet (in 1000s)
y = np.array([2, 4, 5, 4, 5])  # Prices (in $100k)

plt.scatter(X, y, s=100, color='blue')
plt.xlabel('Square Feet (thousands)')
plt.ylabel('Price ($100k)')
plt.title('House Prices vs Square Footage')
plt.grid(True, alpha=0.3)
plt.show()

print(f"We have {len(X)} data points")

### The Goal: Find the Best Line

We want to find a line: **y = mx + b**
- **m** = slope (how much price increases per 1000 sq ft)
- **b** = intercept (base price)

But which line is "best"?

In [ ]:
# Let's try a few random lines
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Three different lines (slope, intercept pairs)
lines = [(0.5, 1), (1, 0.5), (0.8, 1)]

for i, (m, b) in enumerate(lines):
    axes[i].scatter(X, y, s=100, color='blue', zorder=3)
    
    # Plot the line
    x_line = np.array([0, 6])
    y_line = m * x_line + b
    axes[i].plot(x_line, y_line, 'r-', linewidth=2, label=f'y = {m}x + {b}')
    
    # Calculate and show error
    predictions = m * X.flatten() + b
    mse = np.mean((y - predictions)**2)
    
    axes[i].set_xlabel('Square Feet (thousands)')
    axes[i].set_ylabel('Price ($100k)')
    axes[i].set_title(f'Line {i+1}: MSE = {mse:.2f}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim(0, 6)
    axes[i].set_ylim(0, 7)

plt.tight_layout()
plt.show()

print("Which line is best? The one with the lowest MSE!")

## Part 2: The Loss Landscape

For every possible (m, b) combination, we can calculate the MSE. This creates a **loss landscape** - a surface where lower points are better.

Our goal: Find the **lowest point** on this surface!

In [ ]:
# Create a grid of (m, b) values
m_values = np.linspace(-0.5, 2, 100)
b_values = np.linspace(-1, 3, 100)
M, B = np.meshgrid(m_values, b_values)

# Calculate MSE for each (m, b) combination
def calculate_mse(m, b, X, y):
    predictions = m * X.flatten() + b
    return np.mean((y - predictions)**2)

# Vectorized MSE calculation
MSE = np.zeros_like(M)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        MSE[i, j] = calculate_mse(M[i, j], B[i, j], X, y)

# Plot the loss landscape
fig = plt.figure(figsize=(14, 5))

# 3D surface plot
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(M, B, MSE, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Slope (m)')
ax1.set_ylabel('Intercept (b)')
ax1.set_zlabel('MSE (Loss)')
ax1.set_title('Loss Landscape (3D)')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot (top-down view)
ax2 = fig.add_subplot(122)
contour = ax2.contour(M, B, MSE, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.set_xlabel('Slope (m)')
ax2.set_ylabel('Intercept (b)')
ax2.set_title('Loss Landscape (Contour)')
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()

print("\nThe dark blue valley is where the best (m, b) combo lives")
print("But how do we find it without trying every combination?")

## Part 3: The Intuition - Rolling Downhill

**Key Idea:** Imagine you're blindfolded on a hill. How do you get to the bottom?

1. Feel which direction is steepest downhill
2. Take a small step in that direction
3. Repeat until you reach the bottom

This is **Gradient Descent**!

- **Gradient** = the direction of steepest ascent (uphill)
- **Descent** = we go the opposite direction (downhill)

### The Math 
For linear regression: **y = mx + b**

Loss function (MSE): 
$$L(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

The gradients (derivatives) tell us how to update:
$$\frac{\partial L}{\partial m} = -\frac{2}{n} \sum_{i=1}^{n} x_i(y_i - (mx_i + b))$$
$$\frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - (mx_i + b))$$

Update rules:
$$m_{new} = m_{old} - \alpha \cdot \frac{\partial L}{\partial m}$$
$$b_{new} = b_{old} - \alpha \cdot \frac{\partial L}{\partial b}$$

where **α** (alpha) is the **learning rate** (step size)

## Part 4: Implementing Gradient Descent from Scratch

In [ ]:
def gradient_descent(X, y, learning_rate=0.1, n_iterations=300):
    """
    Perform gradient descent to find the best m and b
    
    Parameters:
    - X: input features (already flattened to 1D)
    - y: target values
    - learning_rate: how big of a step to take (alpha)
    - n_iterations: how many steps to take
    
    Returns:
    - m, b: final slope and intercept
    - history: list of (m, b, loss) at each iteration
    """
    # Flatten X if needed
    X_flat = X.flatten()
    n = len(X_flat)
    
    # Initialize m and b to zero
    m = 0.0
    b = 0.0
    
    # Track history for visualization
    history = []
    
    for iteration in range(n_iterations):
        # Make predictions with current m and b
        predictions = m * X_flat + b
        
        # Calculate errors
        errors = y - predictions
        
        # Calculate loss (MSE)
        loss = np.mean(errors**2)
        
        # Calculate gradients
        gradient_m = -(2/n) * np.sum(X_flat * errors)
        gradient_b = -(2/n) * np.sum(errors)
        
        # Update parameters
        m = m - learning_rate * gradient_m
        b = b - learning_rate * gradient_b
        
        # Save history
        history.append((m, b, loss))
        
        # Print progress every 20 iterations
        if iteration % 20 == 0:
            print(f"Iteration {iteration:3d}: m={m:.4f}, b={b:.4f}, Loss={loss:.4f}")
    
    return m, b, history

# Run gradient descent with a more conservative learning rate
print("Starting gradient descent...\n")
m_final, b_final, history = gradient_descent(X, y, learning_rate=0.08, n_iterations=100)

print(f"\n Final result: y = {m_final:.4f}x + {b_final:.4f}")

In [ ]:
# Visualize the result
plt.figure(figsize=(10, 6))
plt.scatter(X, y, s=100, color='blue', label='Data', zorder=3)

# Our gradient descent line
x_line = np.array([0, 6])
y_line_gd = m_final * x_line + b_final
plt.plot(x_line, y_line_gd, 'r-', linewidth=2, label=f'Our GD: y={m_final:.2f}x+{b_final:.2f}')

# Compare to sklearn
model_sklearn = LinearRegression()
model_sklearn.fit(X, y)
y_line_sklearn = model_sklearn.coef_[0] * x_line + model_sklearn.intercept_
plt.plot(x_line, y_line_sklearn, 'g--', linewidth=2, 
         label=f'Sklearn: y={model_sklearn.coef_[0]:.2f}x+{model_sklearn.intercept_:.2f}')

plt.xlabel('Square Feet (thousands)')
plt.ylabel('Price ($100k)')
plt.title('Gradient Descent vs Sklearn')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 6)
plt.ylim(0, 7)
plt.show()

print(f"\nOur gradient descent: m={m_final:.4f}, b={b_final:.4f}")
print(f"Sklearn result:       m={model_sklearn.coef_[0]:.4f}, b={model_sklearn.intercept_:.4f}")
print(f"\n Pretty close! Sklearn uses a more sophisticated method, but gradient descent works!")

## Part 5: Visualizing

Let's see the path gradient descent took to find the solution

In [ ]:
# Extract the path from history
m_path = [h[0] for h in history]
b_path = [h[1] for h in history]
loss_path = [h[2] for h in history]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Loss over iterations
axes[0].plot(loss_path, 'b-', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Decreasing Over Time')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')  # Log scale to see it better

# Right plot: Path on the contour plot
contour = axes[1].contour(M, B, MSE, levels=20, cmap='viridis', alpha=0.6)
axes[1].plot(m_path, b_path, 'ro-', linewidth=2, markersize=4, label='GD Path')
axes[1].plot(m_path[0], b_path[0], 'go', markersize=10, label='Start')
axes[1].plot(m_path[-1], b_path[-1], 'r*', markersize=15, label='End')
axes[1].set_xlabel('Slope (m)')
axes[1].set_ylabel('Intercept (b)')
axes[1].set_title('Gradient Descent Path')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(" Notice:")
print("- Loss decreases monotonically (always going down)")
print("- The path follows the steepest descent at each step")
print("- Eventually reaches the minimum (valley)")

## Part 6: The Learning Rate - Too Fast or Too Slow?

The **learning rate** (α) controls how big our steps are. Let's see what happens with different values.

In [ ]:
# Try different learning rates
learning_rates = [0.005, 0.01, 0.08, 0.3]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, lr in enumerate(learning_rates):
    print(f"\nLearning rate: {lr}")
    m, b, hist = gradient_descent(X, y, learning_rate=lr, n_iterations=100)
    
    # Extract path
    m_p = [h[0] for h in hist]
    b_p = [h[1] for h in hist]
    loss_p = [h[2] for h in hist]
    
    # Plot
    axes[i].contour(M, B, MSE, levels=20, cmap='viridis', alpha=0.4)
    axes[i].plot(m_p, b_p, 'ro-', linewidth=2, markersize=3)
    axes[i].plot(m_p[0], b_p[0], 'go', markersize=10)
    axes[i].plot(m_p[-1], b_p[-1], 'r*', markersize=15)
    axes[i].set_xlabel('Slope (m)')
    axes[i].set_ylabel('Intercept (b)')
    axes[i].set_title(f'Learning Rate = {lr}\nFinal Loss = {loss_p[-1]:.4f}')
    axes[i].set_xlim(-0.5, 2)
    axes[i].set_ylim(-1, 3)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## Part 7: When Does Gradient Descent Stop?

We need **stopping criteria**:

1. **Maximum iterations** - "I've taken enough steps"
2. **Convergence** - "The improvement is tiny now"
3. **Time limit** - "I've spent too long on this"

Let's implement a smarter version:

In [ ]:
def gradient_descent_smart(X, y, learning_rate=0.01, max_iterations=2000, tolerance=1e-6):
    """
    Gradient descent with convergence checking
    
    Stops when:
    - Maximum iterations reached, OR
    - Loss change is less than tolerance
    """
    X_flat = X.flatten()
    n = len(X_flat)
    
    m = 0.0
    b = 0.0
    
    prev_loss = float('inf')
    history = []
    
    for iteration in range(max_iterations):
        predictions = m * X_flat + b
        errors = y - predictions
        loss = np.mean(errors**2)
        
        # Check for convergence
        if abs(prev_loss - loss) < tolerance:
            print(f"\n Converged at iteration {iteration}!")
            print(f"   Loss change: {abs(prev_loss - loss):.2e} < tolerance: {tolerance:.2e}")
            break
        
        gradient_m = -(2/n) * np.sum(X_flat * errors)
        gradient_b = -(2/n) * np.sum(errors)
        
        m = m - learning_rate * gradient_m
        b = b - learning_rate * gradient_b
        
        history.append((m, b, loss))
        prev_loss = loss
        
        if iteration % 100 == 0:
            print(f"Iteration {iteration:4d}: Loss={loss:.6f}")
    
    return m, b, history

# Run it
print("Running gradient descent with convergence checking...\n")
m_smart, b_smart, hist_smart = gradient_descent_smart(X, y, learning_rate=0.01, tolerance=1e-6)

print(f"\nFinal result: y = {m_smart:.6f}x + {b_smart:.6f}")
print(f"Total iterations: {len(hist_smart)}")

## Part 8: Real-World Considerations

### 1. Feature Scaling Matters

When features have very different scales, gradient descent can be slow!

In [ ]:
# Example: house prices with different scales
# X1: square feet (1000-5000)
# X2: number of bedrooms (1-5)

np.random.seed(42)
n_samples = 50
X1 = np.random.uniform(1000, 5000, n_samples)  # Large scale
X2 = np.random.uniform(1, 5, n_samples)        # Small scale
y_multi = 100 * X1 + 50000 * X2 + np.random.normal(0, 50000, n_samples)

print("Feature scales:")
print(f"X1 (sq ft):  mean={X1.mean():.1f}, std={X1.std():.1f}")
print(f"X2 (bedrooms): mean={X2.mean():.1f}, std={X2.std():.1f}")
print("\nProblem: X1 is ~1000x larger than X2")
print("Solution: Scale features before gradient descent (we'll learn this soon!)")

### 2. Local vs Global Minima

Our simple problem has **one** minimum (convex).

But some problems have **multiple** minima (non-convex):
- Gradient descent might get stuck in a local minimum
- Neural networks face this problem!
- Solutions: random restarts, momentum, adaptive learning rates

In [ ]:
# Visualize a non-convex function (example)
x = np.linspace(-3, 3, 100)
y_nonconvex = x**4 - 5*x**2 + 4*x

plt.figure(figsize=(10, 6))
plt.plot(x, y_nonconvex, 'b-', linewidth=2)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Parameter')
plt.ylabel('Loss')
plt.title('Non-Convex Loss Function (Multiple Minima)')
plt.grid(True, alpha=0.3)

# Mark the minima
plt.plot(-1.8, -12.5, 'ro', markersize=10, label='Local minimum')
plt.plot(1.4, 0, 'go', markersize=10, label='Global minimum')
plt.legend()
plt.show()

print("  For non-convex functions:")
print("- Starting point matters")
print("- Might need multiple random starts")
print("- Linear regression is nice: always convex!")

## Part 9: Variants of Gradient Descent

What we implemented is **Batch Gradient Descent** - uses all data points at each step.

Other variants:

### 1. Stochastic Gradient Descent (SGD)
- Use ONE random data point at a time
- Much faster per iteration
- More noisy, but can escape local minima
- Good for huge datasets

### 2. Mini-batch Gradient Descent
- Use a small batch (e.g., 32 samples) at a time
- Best of both worlds
- Standard for neural networks

### 3. Advanced Optimizers
- **Momentum**: Remember past gradients, build speed
- **Adam**: Adaptive learning rates per parameter
- **RMSprop**: Another adaptive method
- You'll see these in pyMAISE!

## Part 10: Summary & Key Takeaways

### What We Learned

1. **The Goal**: Find parameters that minimize loss

2. **The Algorithm**: Gradient Descent
   - Start with random parameters
   - Calculate gradient (direction of steepest ascent)
   - Move opposite direction (descent)
   - Repeat until convergence

3. **Key Hyperparameter**: Learning Rate (α)
   - Too small → slow convergence
   - Too large → oscillation or divergence
   - Typical values: 0.001 - 0.1

4. **When to Stop**: 
   - Maximum iterations
   - Loss change < tolerance
   - Validation loss stops improving

### Why This Matters

- **Every** ML model uses some form of gradient descent
- Understanding it helps you:
  - Debug training problems
  - Choose hyperparameters wisely
  - Understand why models fail
  - Appreciate what happens when you call `.fit()`

### What's Next

- Regularization (preventing overfitting during GD)
- Classification (GD for logistic regression)
- Neural networks (GD on steroids with backpropagation)
- **pyMAISE**: Will use these optimizers under the hood

## Practice Problems if we have time

Try these to solidify your understanding:

### Problem 1: Tune the Learning Rate
Run gradient descent with learning rates [0.001, 0.05, 0.08, 0.15, .3]. Which gives the best result in 50 iterations?

### Problem 2: Different Starting Points
Modify the gradient descent function to start at (m=2, b=2) instead of (0, 0). Does it still converge to the same place?

### Problem 3: Monitoring Progress
Plot both the loss AND the parameters (m, b) over iterations. Do they stabilize at the same time?

### Problem 4: Noisy Data
Add more noise to the y values (e.g., `y = y + np.random.normal(0, 2, len(y))`). Does gradient descent still work? Is the final loss higher or lower?